# training ANPEs on adroit

In [1]:
import os, sys 
import numpy as np

import torch
from torch import nn 
from torch.utils.tensorboard.writer import SummaryWriter

from sbi import utils as Ut
from sbi import inference as Inference

from sedflow import data as D
from sedflow import util as U
from sedflow import flows as F

In [2]:
cuda = torch.cuda.is_available()
device = ("cuda:0" if cuda else "cpu")

seed = 12387
torch.manual_seed(seed)
if cuda:
    torch.cuda.manual_seed(seed)

In [3]:
bands = 'grzW1W2'
freez = False

In [4]:
x_train, y_train = D.load_modelb('train', bands=bands, infer_redshift=freez)
print('Ntrain + Nvalid = %i' % (x_train.shape[0]))

Ntrain + Nvalid = 1000000


In [5]:
x_train = x_train[:10000]
y_train = y_train[:10000]

In [6]:
prior_low   = [6, 0., 0., 0., 0., 1e-2, np.log10(4.5e-5), np.log10(4.5e-5), 0, 0., -2., 0.1, 0.0, 0.1]
prior_high  = [13.0, 1., 1., 1., 1., 13.27, np.log10(1.5e-2), np.log10(1.5e-2), 3., 3., 1., 15., 0.15, 0.7]

lower_bounds = torch.tensor(prior_low).to(device)
upper_bounds = torch.tensor(prior_high).to(device)

#prior = Ut.BoxUniform(low=lower_bounds, high=upper_bounds, device=device)

In [7]:
_x_train = np.empty(x_train.shape)
for i in range(x_train.shape[1]): 
    _x_train[:,i] = U.inv_cdf_transform(x_train[:,i], [prior_low[i], prior_high[i]])

In [8]:
neural_posterior = Ut.posterior_nn('maf', 
        hidden_features=500, 
        num_transforms=10,   
        use_batch_norm=True)

In [10]:
_prior = Ut.BoxUniform(low=torch.tensor([-10. for i in range(len(prior_low))]),
                       high=torch.tensor([10. for i in range(len(prior_high))]),
                       device=device)
anpe = Inference.SNPE(prior=_prior,
        density_estimator=neural_posterior,
        device=device)

/home/chhahn/.conda/envs/sbi/lib/python3.10/site-packages/sbi/utils/torchutils.py:27: UserWarning: GPU was selected as a device for training the neural network. Note that we expect no significant speed ups in training for the default architectures we provide. Using the GPU will be effective only for large neural networks with operations that are fast on the GPU, e.g., for a CNN or RNN `embedding_net`.
  warnings.warn(


In [11]:
anpe.append_simulations(
    torch.as_tensor(_x_train.astype(np.float32)).to(device),
    torch.as_tensor(y_train.astype(np.float32)).to(device))

In [12]:
p_x_y_est = anpe.train()

 Neural network successfully converged after 34 epochs.

In [13]:
anpe._summary['best_validation_log_prob']

[-20.182760009765627]

In [8]:
flo = F.Flow(device=device)
flo.set_prior('uniform', low=prior_low, high=prior_high)

nde = flo.nde(
        architecture='maf',
        n_hidden=500,
        n_transf=10,
        n_blocks=10,
        p_drop=0.1,
        use_batch_norm=True)

flo.train_flow(x_train, y_train,
            nde=nde, # nde we're training
            training_batch_size=500)

/home/chhahn/.conda/envs/sbi/lib/python3.10/site-packages/sbi/utils/torchutils.py:27: UserWarning: GPU was selected as a device for training the neural network. Note that we expect no significant speed ups in training for the default architectures we provide. Using the GPU will be effective only for large neural networks with operations that are fast on the GPU, e.g., for a CNN or RNN `embedding_net`.
  warnings.warn(


 Neural network successfully converged after 85 epochs.
        -------------------------
        ||||| ROUND 1 STATS |||||:
        -------------------------
        Epochs trained: 85
        Best validation performance: -15.9884
        -------------------------
        


In [11]:
flo.anpe._summary#['best_validation_log_prob']

{'epochs_trained': [85],
 'best_validation_log_prob': [-15.98836962890625],
 'validation_log_probs': [-54.62312890625,
  -32.7454833984375,
  -24.96708984375,
  -22.619412109375,
  -20.699833984375,
  -20.082439453125,
  -19.206560546875,
  -19.1335712890625,
  -18.90255859375,
  -18.978025390625,
  -18.56115625,
  -18.2855859375,
  -18.416763671875,
  -18.13968359375,
  -18.0872958984375,
  -17.9186650390625,
  -18.007107421875,
  -17.8262666015625,
  -17.79817578125,
  -17.5327802734375,
  -17.73228125,
  -17.471955078125,
  -17.39033984375,
  -17.127640625,
  -17.3000732421875,
  -17.17801171875,
  -17.067291015625,
  -17.251755859375,
  -17.2812412109375,
  -17.0622314453125,
  -16.8802001953125,
  -16.93938671875,
  -16.853935546875,
  -16.9089619140625,
  -16.72507421875,
  -16.564611328125,
  -16.6731396484375,
  -16.4977138671875,
  -16.7696240234375,
  -16.5911015625,
  -16.501736328125,
  -16.677783203125,
  -16.53321630859375,
  -16.6384072265625,
  -16.5022880859375,
  -16.